In [2]:
from pathlib import Path
import pandas as pd

file_path = Path(
    r"C:\Users\hp\Downloads\archive (3)"
    r"\healthcare_appointment_no_show_wait_time.csv"
)

no_show_data = pd.read_csv(file_path)

print("Total records:", len(no_show_data))
print("Columns:", no_show_data.columns.tolist())

display(
    no_show_data[
        [
            "department",
            "appointment_type",
            "scheduled_hour",
            "waiting_time_minutes",
            "previous_no_shows",
            "appointment_status",
        ]
    ].head()
)

print("\nMissing values:")
print(no_show_data.isna().sum())

print("\nAppointment status:")
print(no_show_data["appointment_status"].value_counts())

print("\nWait-time summary:")
print(no_show_data["waiting_time_minutes"].describe())

Total records: 2800
Columns: ['appointment_id', 'appointment_date', 'patient_age', 'gender', 'department', 'appointment_type', 'scheduled_hour', 'waiting_time_minutes', 'reminder_sent', 'previous_no_shows', 'appointment_status']


,department,appointment_type,scheduled_hour,waiting_time_minutes,previous_no_shows,appointment_status
0,Orthopedics,New,12,82,4,Completed
1,Cardiology,Follow-up,9,176,2,No-Show
2,Orthopedics,New,13,100,2,Completed
3,Orthopedics,New,14,54,4,Completed
4,Pediatrics,New,8,121,3,Completed



Missing values:
appointment_id          0
appointment_date        0
patient_age             0
gender                  0
department              0
appointment_type        0
scheduled_hour          0
waiting_time_minutes    0
reminder_sent           0
previous_no_shows       0
appointment_status      0
dtype: int64

Appointment status:
appointment_status
Completed    1878
No-Show       646
Cancelled     276
Name: count, dtype: int64

Wait-time summary:
count    2800.000000
mean       90.189286
std        50.522952
min         5.000000
25%        46.750000
50%        88.000000
75%       133.000000
max       179.000000
Name: waiting_time_minutes, dtype: float64


In [4]:
completed = no_show_data[
    no_show_data["appointment_status"].eq("Completed")
].copy()

print("Completed appointments:", len(completed))

print("\nWait time by appointment status:")
display(
    no_show_data.groupby(
        "appointment_status"
    )["waiting_time_minutes"]
    .agg(["count", "mean", "median"])
    .round(2)
)

print("\nCompleted wait-time summary:")
print(completed["waiting_time_minutes"].describe())

Completed appointments: 1878

Wait time by appointment status:


,count,mean,median
appointment_status,,,
Cancelled,276,88.81,88.0
Completed,1878,89.95,88.0
No-Show,646,91.48,88.0



Completed wait-time summary:
count    1878.000000
mean       89.947284
std        50.559332
min         5.000000
25%        46.250000
50%        88.000000
75%       133.000000
max       179.000000
Name: waiting_time_minutes, dtype: float64


In [6]:
wait_by_department = (
    completed.groupby("department")[
        "waiting_time_minutes"
    ]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
    .round(2)
)

wait_by_hour = (
    completed.groupby("scheduled_hour")[
        "waiting_time_minutes"
    ]
    .agg(["count", "mean", "median"])
    .round(2)
)

print("WAIT TIME BY DEPARTMENT")
display(wait_by_department)

print("WAIT TIME BY SCHEDULED HOUR")
display(wait_by_hour)

WAIT TIME BY DEPARTMENT


,count,mean,median
department,,,
Cardiology,403,93.05,94.0
Orthopedics,353,91.84,88.0
Pediatrics,381,89.97,89.0
General Medicine,371,88.53,87.0
Dermatology,370,86.16,82.0


WAIT TIME BY SCHEDULED HOUR


,count,mean,median
scheduled_hour,,,
8,154,90.66,95.5
9,136,93.51,93.0
10,158,85.39,74.5
11,152,86.84,91.0
12,163,86.94,82.0
13,151,93.83,99.0
14,153,87.32,80.0
15,160,94.95,94.0
16,161,87.83,86.0


In [8]:
wait_by_type = (
    completed.groupby("appointment_type")[
        "waiting_time_minutes"
    ]
    .agg(["count", "mean", "median"])
    .round(2)
)

completed["no_show_history_group"] = pd.cut(
    completed["previous_no_shows"],
    bins=[-1, 0, 1, 2, float("inf")],
    labels=[
        "No history",
        "1 previous no-show",
        "2 previous no-shows",
        "3+ previous no-shows",
    ],
)

wait_by_no_show_history = (
    completed.groupby(
        "no_show_history_group",
        observed=True,
    )["waiting_time_minutes"]
    .agg(["count", "mean", "median"])
    .round(2)
)

print("WAIT TIME BY APPOINTMENT TYPE")
display(wait_by_type)

print("WAIT TIME BY PREVIOUS NO-SHOW HISTORY")
display(wait_by_no_show_history)

WAIT TIME BY APPOINTMENT TYPE


,count,mean,median
appointment_type,,,
Follow-up,930,88.88,87.0
New,948,91.00,89.5


WAIT TIME BY PREVIOUS NO-SHOW HISTORY


,count,mean,median
no_show_history_group,,,
No history,376,92.30,91.0
1 previous no-show,383,86.84,86.0
2 previous no-shows,387,88.87,87.0
3+ previous no-shows,732,90.94,89.0


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

model_data = completed[
    [
        "department",
        "scheduled_hour",
        "appointment_type",
        "previous_no_shows",
        "waiting_time_minutes",
    ]
].dropna()

X = model_data.drop(
    columns="waiting_time_minutes"
)

y = model_data["waiting_time_minutes"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

baseline_prediction = [y_train.mean()] * len(y_test)

print(
    "Baseline MAE:",
    round(
        mean_absolute_error(
            y_test,
            baseline_prediction,
        ),
        2,
    ),
    "minutes",
)

print(
    "Baseline RMSE:",
    round(
        mean_squared_error(
            y_test,
            baseline_prediction,
        ) ** 0.5,
        2,
    ),
    "minutes",
)

print(
    "Baseline R²:",
    round(
        r2_score(
            y_test,
            baseline_prediction,
        ),
        4,
    ),
)

Baseline MAE: 44.14 minutes
Baseline RMSE: 51.8 minutes
Baseline R²: -0.001


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    "scheduled_hour",
    "previous_no_shows",
]

categorical_features = [
    "department",
    "appointment_type",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent",
                        ),
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                        ),
                    ),
                ],
            ),
            categorical_features,
        ),
    ],
)

patient_flow_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=250,
                learning_rate=0.06,
                max_leaf_nodes=15,
                l2_regularization=1.0,
                random_state=42,
            ),
        ),
    ],
)

patient_flow_model.fit(X_train, y_train)

predictions = patient_flow_model.predict(X_test)

print(
    "Training MAE:",
    round(
        mean_absolute_error(
            y_train,
            patient_flow_model.predict(X_train),
        ),
        2,
    ),
    "minutes",
)

print(
    "Testing MAE:",
    round(
        mean_absolute_error(y_test, predictions),
        2,
    ),
    "minutes",
)

print(
    "Testing RMSE:",
    round(
        mean_squared_error(y_test, predictions) ** 0.5,
        2,
    ),
    "minutes",
)

print(
    "Testing R²:",
    round(r2_score(y_test, predictions), 4),
)

Training MAE: 39.44 minutes
Testing MAE: 46.37 minutes
Testing RMSE: 54.69 minutes
Testing R²: -0.116


## Conclusion: No-Show and Wait-Time Dataset

- The dataset contains 2,800 synthetic appointments; 1,878 completed appointments were used for wait-time analysis.
- Cancelled and no-show records also contained wait-time values, so only completed appointments were retained for safer modeling.
- Department and scheduled hour showed small descriptive differences, while appointment type and no-show history were weak signals.
- The wait-time baseline MAE was 44.14 minutes.
- HistGradientBoosting performed worse than the baseline: testing MAE 46.37 minutes and R² -0.1100.
- This dataset is rejected for SmartCare wait-time model training.
- No model is saved, deployed, or merged into SmartCare from this dataset.